In [3]:
# Load the different transcriptome files. 
library(tidyverse)
library(data.table)

# Define the file path and cell lines
file_path <- "/mnt/dawnccle2/nf_rnasplice_CCLE/rmats_nostat/"
cell_lines <- c("JJN3", "T47D", "NOMO1", "HCC38", "TOU21G", "KELLY", 
                "GI1", "MCF7", "JHH6", "MONOMAC6", "HEK")

# Create a list to store the data frames
transcriptome_list <- list()

# Read each file and add folder name as a column
for (cell_line in cell_lines) {
  file <- file.path(file_path, cell_line, "SE.MATS.JC.txt")
  if (file.exists(file)) {
    df <- read.delim(file, sep = "\t")
    # Remove duplicate columns
    df <- df[, !duplicated(colnames(df))]
    df$Folder <- cell_line
    transcriptome_list[[cell_line]] <- df
  }
}

# Combine all data frames
transcriptome_data <- bind_rows(transcriptome_list) %>%
  select(-IncLevel2, -IJC_SAMPLE_2, -SJC_SAMPLE_2, -IncLevelDifference) %>%
  filter(!is.na(IncLevel1)) %>%
  mutate(ID = paste(chr, exonStart_0base, exonEnd, upstreamES, upstreamEE, downstreamES, downstreamEE, sep = "-")) %>% 
  mutate(NameConcat = paste(GeneID, geneSymbol, ID, sep=";"))

# Write output file
write_tsv(transcriptome_data, "/mnt/dawnccle2/processed_data/transcriptome_deepseq/ALL_deepseq_SE.MATS.JC.tsv")

── Attaching core tidyverse packages ──────────────────────── tidyverse 2.0.0 ──
✔ dplyr     1.1.4     ✔ readr     2.1.5
✔ forcats   1.0.0     ✔ stringr   1.5.1
✔ ggplot2   3.5.1     ✔ tibble    3.2.1
✔ lubridate 1.9.4     ✔ tidyr     1.3.1
✔ purrr     1.0.4     
── Conflicts ────────────────────────────────────────── tidyverse_conflicts() ──
✖ dplyr::filter() masks stats::filter()
✖ dplyr::lag()    masks stats::lag()
ℹ Use the conflicted package (<http://conflicted.r-lib.org/>) to force all conflicts to become errors

Attaching package: ‘data.table’


The following objects are masked from ‘package:lubridate’:

    hour, isoweek, mday, minute, month, quarter, second, wday, week,
    yday, year


The following objects are masked from ‘package:dplyr’:

    between, first, last


The following object is masked from ‘package:purrr’:

    transpose




In [4]:
head(transcriptome_data)

,ID,GeneID,geneSymbol,chr,strand,exonStart_0base,exonEnd,upstreamES,upstreamEE,downstreamES,⋯,ID.1,IJC_SAMPLE_1,SJC_SAMPLE_1,IncFormLen,SkipFormLen,PValue,FDR,IncLevel1,Folder,NameConcat
,<chr>,<chr>,<chr>,<chr>,<chr>,<int>,<int>,<int>,<int>,<int>,⋯,<int>,<chr>,<chr>,<int>,<int>,<lgl>,<lgl>,<chr>,<chr>,<chr>
1,chrY-24784136-24784208-24774349-24774421-24788893-24788965,ENSG00000187191.15,DAZ3,chrY,-,24784136,24784208,24774349,24774421,24788893,⋯,0,"0,0","0,0",271,199,NA,NA,"NA,NA",JJN3,ENSG00000187191.15;DAZ3;chrY-24784136-24784208-24774349-24774421-24788893-24788965
2,chrY-24786513-24786585-24784136-24784208-24788893-24788965,ENSG00000187191.15,DAZ3,chrY,-,24786513,24786585,24784136,24784208,24788893,⋯,1,"0,0","0,0",271,199,NA,NA,"NA,NA",JJN3,ENSG00000187191.15;DAZ3;chrY-24786513-24786585-24784136-24784208-24788893-24788965
3,chrY-24791270-24791342-24788893-24788965-24793649-24793721,ENSG00000187191.15,DAZ3,chrY,-,24791270,24791342,24788893,24788965,24793649,⋯,2,"0,0","0,0",271,199,NA,NA,"NA,NA",JJN3,ENSG00000187191.15;DAZ3;chrY-24791270-24791342-24788893-24788965-24793649-24793721
4,chrY-24796026-24796098-24793649-24793721-24798422-24798494,ENSG00000187191.15,DAZ3,chrY,-,24796026,24796098,24793649,24793721,24798422,⋯,3,"0,0","0,0",271,199,NA,NA,"NA,NA",JJN3,ENSG00000187191.15;DAZ3;chrY-24796026-24796098-24793649-24793721-24798422-24798494
5,chrY-24628221-24628339-24626043-24626162-24631193-24631299,ENSG00000183795.8,BPY2B,chrY,+,24628221,24628339,24626043,24626162,24631193,⋯,4,"0,0","0,0",317,199,NA,NA,"NA,NA",JJN3,ENSG00000183795.8;BPY2B;chrY-24628221-24628339-24626043-24626162-24631193-24631299
6,chrY-23236827-23236899-23234431-23234503-23241605-23241677,ENSG00000205944.11,DAZ2,chrY,+,23236827,23236899,23234431,23234503,23241605,⋯,5,"0,0","0,0",271,199,NA,NA,"NA,NA",JJN3,ENSG00000205944.11;DAZ2;chrY-23236827-23236899-23234431-23234503-23241605-23241677


In [5]:
calculate_ratio <- function(I, S) {
  I_values <- as.numeric(unlist(strsplit(I, ",")))
  S_values <- as.numeric(unlist(strsplit(S, ",")))
  ratio <- I_values / (I_values + S_values)
  return(paste(round(ratio,3), collapse = ","))
}

calculate_average <- function(PSI){
  PSI_values <- as.numeric(unlist(strsplit(PSI, ",")))
  average <- mean(PSI_values)
  return(round(average, 3))
}

calculate_average_count_sum <- function(I, S){
  I_values <- as.numeric(unlist(strsplit(I, ",")))
  S_values <- as.numeric(unlist(strsplit(S, ",")))
  total_sum <- I_values + S_values
  average_count_sum <- mean(total_sum)
  return(round(average_count_sum, 0))
}

# Apply the function to the data frame
transcriptome_data <- transcriptome_data %>%
  mutate(PSI1 = mapply(calculate_ratio, IJC_SAMPLE_1, SJC_SAMPLE_1)) %>% 
  mutate(PSI1_average = mapply(calculate_average, PSI1)) %>% 
  mutate(average_count_sum = mapply(calculate_average_count_sum, IJC_SAMPLE_1, SJC_SAMPLE_1))


In [6]:
# Filter for high count sum
transcriptome_data_filtered <- transcriptome_data %>% 
    filter(average_count_sum >= 30) 

In [ ]:
transcriptome_pivot <- transcriptome_data_filtered %>% select(NameConcat,Folder, PSI1_average) %>% 
  pivot_wider(names_from = Folder, values_from = PSI1_average) 

write_tsv(transcriptome_pivot, "/mnt/dawnccle2/processed_data/transcriptome_deepseq/ALL_deepseq_SE.MATS.JC_pivot.tsv")

# Now also read in the MPRA data

In [ ]:
mpra_cts <- read_csv("/mnt/dawnccle2/melange/figures_outputs/fig02/fig02_num_cell_type_celltype_specific.csv") %>% 
separate(index_offset, into = c("index", "offset"), sep = "__", remove = F)  

head(mpra_cts)

Rows: 1147 Columns: 13
── Column specification ────────────────────────────────────────────────────────
Delimiter: ","
chr (4): index_offset, max_sample, min_sample, target_cell_type
dbl (9): upsilon, upsilon_reverse, tau, tau_reverse, num_na_per_row, row_max...

ℹ Use `spec()` to retrieve the full column specification for this data.
ℹ Specify the column types or set `show_col_types = FALSE` to quiet this message.


index_offset,index,offset,upsilon,upsilon_reverse,tau,tau_reverse,num_na_per_row,row_max,row_min,row_mean,row_median,max_sample,min_sample,target_cell_type
<chr>,<chr>,<chr>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<chr>,<chr>,<chr>
ENSG00000106066.15;CPVL;chr7-29066022-29066121-29064060-29064234-29071772-29071904__0:-65:0,ENSG00000106066.15;CPVL;chr7-29066022-29066121-29064060-29064234-29071772-29071904,0:-65:0,0.9713060,0.05810579,0.9713060,0.05810579,8,1.0000000,0.000000000,0.05644562,0.01660628,Kelly,COLO783,Kelly
ENSG00000138821.14;SLC39A8;chr4-102285948-102285973-102267871-102268079-102304316-102304481__0:0:0,ENSG00000138821.14;SLC39A8;chr4-102285948-102285973-102267871-102268079-102304316-102304481,0:0:0,0.9344682,0.05526994,0.9638482,0.05530171,0,0.9408392,0.001148387,0.05510198,0.01768707,Kelly,COLO783,Kelly
ENSG00000067048.17;DDX3Y;chrY-12916516-12916634-12916260-12916442-12916906-12917060__100:0:0,ENSG00000067048.17;DDX3Y;chrY-12916516-12916634-12916260-12916442-12916906-12917060,100:0:0,0.9036632,0.06814832,0.9488464,0.06814832,0,0.9090909,0.000000000,0.06656347,0.02992148,GI1,DAOY,GI1
ENSG00000189212.12;DPY19L2P1;chr7-35123997-35124078-35121506-35121608-35132309-35132369__0:-71:0,ENSG00000189212.12;DPY19L2P1;chr7-35123997-35124078-35121506-35121608-35132309-35132369,0:-71:0,0.8925576,0.13646939,0.8925576,0.13656002,9,1.0000000,0.001326366,0.13369411,0.07162109,Kelly,HEK,Kelly
ENSG00000039123.16;MTREX;chr5-55341680-55341771-55340009-55340184-55343330-55343372__0:-81:0,ENSG00000039123.16;MTREX;chr5-55341680-55341771-55340009-55340184-55343330-55343372,0:-81:0,0.8874820,0.10545485,0.9152414,0.10560808,9,0.9411255,0.002897606,0.10510256,0.06400995,Kelly,HEK,Kelly
ENSG00000158796.17;DEDD;chr1-161130814-161130843-161124137-161124526-161132550-161132667__0:0:0,ENSG00000158796.17;DEDD;chr1-161130814-161130843-161124137-161124526-161132550-161132667,0:0:0,0.8745905,0.14550095,0.8745905,0.14615217,7,1.0000000,0.008871890,0.14970365,0.11028612,Kelly,A375,Kelly


In [ ]:
# # Join with transcriptome_pivot by index in mpra_cts and NameConcat in transcriptome_pivot
# mpra_cts_transcriptome <- mpra_cts %>% 
#   left_join(transcriptome_pivot, by = c("index" = "NameConcat"))

# mpra_cts_transcriptome %>% filter(target_cell_type == "Kelly")

unique_cell_types <- unique(mpra_cts$target_cell_type)

for (cell_type in unique_cell_types) {
  mpra_cts_index <- mpra_cts %>% 
    filter(target_cell_type == cell_type) %>% 
    pull(index)

  mpra_cts_transcriptome_celltype <- transcriptome_pivot %>% 
    filter(NameConcat %in% mpra_cts_index)

  ggplot(mpra_cts_transcriptome_celltype, aes(x = mpra_cts, y = PSI1_average)) +
    geom_point() +
    geom_smooth(method = "lm") +
    labs(x = "MPRA CTS", y = "Transcriptome PSI")
}

[1] "Kelly"
[1] "GI1"
[1] "IPC298"
[1] "K562"
[1] "KMRC1"
[1] "T47D"
[1] "786O"
[1] "COLO783"
[1] "A172"
[1] "MEL202"
[1] "DAOY"
[1] "OC316"
[1] "KMRC20"
[1] "SNU398"
[1] "SNU423"
[1] "GAMG"
[1] "8MGBA"
[1] "769P"
[1] "A375"
[1] "TOV21G"
[1] "EFO27"
[1] "JHH6"
[1] "HEK"
[1] "HCC38"
[1] "SNU449"
[1] "MDAMB231"
[1] "SKNAS"
[1] "IGR37"
[1] "MeWo"
[1] "PLCPRF5"
[1] "MELHO"
[1] "CAL120"
[1] "GB1"
[1] "HCC1428"
[1] "DBTRG05MG"
[1] "SF126"
[1] "ACHN"
